# Model Evaluation and Operating Point Selection

This notebook evaluates the trained landslide model using a spatially held-out test split.
It reports:
- ROC-AUC
- PR-AUC
- recall/precision tradeoff
- validation threshold tuning
- final model operating point for deployment

The goal is to select a threshold that balances missed landslides against false alarms.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
    precision_recall_curve,
)

PROJECT_DIR = Path("..").resolve()
PROC_DIR = PROJECT_DIR / "data" / "processed"
MODEL_DIR = PROJECT_DIR / "outputs" / "models"
METRIC_DIR = PROJECT_DIR / "outputs" / "metrics"
FIG_DIR = PROJECT_DIR / "outputs" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
METRIC_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = "cpu"
print("Project directories ready.")

In [ ]:
X = np.load(PROC_DIR / "X.npy").astype(np.float32)
y = np.load(PROC_DIR / "y.npy").astype(np.float32)
index = pd.read_csv(PROC_DIR / "dataset_index.csv")

val_mask = (index["split"] == "val").to_numpy()
test_mask = (index["split"] == "test").to_numpy()

print("X shape:", X.shape)
print("y positives:", int(y.sum()))
print("validation rows:", int(val_mask.sum()))
print("test rows:", int(test_mask.sum()))

In [ ]:
# Example CNN model definition
class ConvBlock(nn.Module):
    def __init__(self, cin, cout):
        super().__init__()
        self.c1 = nn.Conv2d(cin, cout, 3, padding=1, bias=False)
        self.b1 = nn.BatchNorm2d(cout)
        self.c2 = nn.Conv2d(cout, cout, 3, padding=1, bias=False)
        self.b2 = nn.BatchNorm2d(cout)

    def forward(self, x):
        x = F.relu(self.b1(self.c1(x)))
        x = F.relu(self.b2(self.c2(x)))
        return F.max_pool2d(x, 2)

class LandslideCNN(nn.Module):
    def __init__(self, in_ch=8, dropout=0.5):
        super().__init__()
        self.blocks = nn.Sequential(
            ConvBlock(in_ch, 16),
            ConvBlock(16, 32),
            ConvBlock(32, 64),
            ConvBlock(64, 128),
        )
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Dropout(dropout),
            nn.Linear(128, 1),
        )

    def forward(self, x):
        return self.head(self.blocks(x)).squeeze(1)

with open(PROC_DIR / "norm_stats.json", "r") as f:
    stats = json.load(f)

CHANNELS = stats["channels"]
model = LandslideCNN(len(CHANNELS))
model.load_state_dict(torch.load(MODEL_DIR / "cnn_full.pt", map_location=DEVICE))
model.eval()

print("CNN loaded successfully:", len(CHANNELS), "input channels")

In [ ]:
@torch.no_grad()
def predict(mask, batch_size=128):
    idx = np.where(mask)[0]
    out = []
    for i in range(0, len(idx), batch_size):
        batch_idx = idx[i:i + batch_size]
        xb = torch.from_numpy(X[batch_idx]).float()
        probs = torch.sigmoid(model(xb)).numpy()
        out.append(probs)
    return np.concatenate(out)

p_val = predict(val_mask)
p_test = predict(test_mask)
y_val = y[val_mask]
y_test = y[test_mask]

print("Validation ROC-AUC:", roc_auc_score(y_val, p_val))
print("Test ROC-AUC:", roc_auc_score(y_test, p_test))
print("Test PR-AUC:", average_precision_score(y_test, p_test))

In [ ]:
thresholds = np.linspace(0.05, 0.95, 181)

rows = []
for t in thresholds:
    pred = (p_val >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_val, pred, labels=[0, 1]).ravel()
    rows.append({
        "threshold": float(t),
        "recall": tp / (tp + fn) if (tp + fn) else 0.0,
        "precision": tp / (tp + fp) if (tp + fp) else 0.0,
        "specificity": tn / (tn + fp) if (tn + fp) else 0.0,
        "fn": int(fn),
        "fp": int(fp),
    })

sweep = pd.DataFrame(rows)
sweep["fpr"] = 1.0 - sweep["specificity"]
sweep["youden"] = sweep["recall"] + sweep["specificity"] - 1.0

print(sweep.head())
print("\nBest Youden J threshold candidate:")
print(sweep.loc[sweep["youden"].idxmax(), ["threshold", "recall", "specificity", "youden"]])

In [ ]:
best_row = sweep.loc[sweep["youden"].idxmax()]
THRESH = float(best_row["threshold"])

print("Chosen operating threshold:", THRESH)
print("Validation recall:", best_row["recall"])
print("Validation specificity:", best_row["specificity"])